# Mô Phỏng Máy Hút Bụi (Vacuum Cleaner Agent)

**Quy ước ma trận:**
- `0` → Ô trống / Máy hút bụi
- `1` → Bụi
- `2` → Tường / Vật cản

In [ ]:
import numpy as np
import random

In [ ]:
# ── Cấu hình ──
ROWS      = 5
COLS      = 7
WALL_PROB = 0.15
DUST_PROB = 0.35
MAX_STEPS = 300

# ── Tạo môi trường ngẫu nhiên ──
def create_env(rows, cols, wall_prob, dust_prob):
    grid = np.zeros((rows, cols), dtype=int)
    for r in range(rows):
        for c in range(cols):
            v = random.random()
            if v < wall_prob:
                grid[r][c] = 2
            elif v < wall_prob + dust_prob:
                grid[r][c] = 1
    free = [(r, c) for r in range(rows) for c in range(cols) if grid[r][c] != 2]
    pos = random.choice(free)
    return grid, pos

grid, start = create_env(ROWS, COLS, WALL_PROB, DUST_PROB)
total_dust = int(np.sum(grid == 1))

print(f"Ma trận ban đầu (vị trí máy: {start}):")
print(grid)
print(f"Tổng bụi: {total_dust} ô")

In [ ]:
# ── Agent ──
MOVES = {'UP': (-1,0), 'DOWN': (1,0), 'LEFT': (0,-1), 'RIGHT': (0,1)}

def run_agent(grid_in, start, max_steps):
    grid = grid_in.copy()
    rows, cols = grid.shape
    pos = list(start)
    history = set()
    steps = 0
    cleaned = 0

    def valid(r, c):
        return 0 <= r < rows and 0 <= c < cols and grid[r][c] != 2

    while steps < max_steps:
        r, c = pos

        if grid[r][c] == 1:
            state = ((r, c), 'CLEAN')
            if state in history:
                return grid, steps, cleaned, 'THAT BAI', 'Lap lai hanh dong CLEAN tai ' + str((r, c))
            history.add(state)
            grid[r][c] = 0
            cleaned += 1
            steps += 1
            if cleaned == total_dust:
                return grid, steps, cleaned, 'THANH CONG', 'Da hut sach toan bo bui'
            continue

        options = [d for d, (dr, dc) in MOVES.items() if valid(r+dr, c+dc)]
        if not options:
            return grid, steps, cleaned, 'THAT BAI', 'Bi ket, khong con huong di hop le'

        direction = random.choice(options)
        state = ((r, c), direction)
        if state in history:
            return grid, steps, cleaned, 'THAT BAI', f'Lap lai hanh dong {direction} tai {(r, c)}'
        history.add(state)

        dr, dc = MOVES[direction]
        pos = [r+dr, c+dc]
        steps += 1

    return grid, steps, cleaned, 'THAT BAI', f'Vuot qua gioi han {max_steps} buoc'


final_grid, steps, cleaned, status, reason = run_agent(grid, start, MAX_STEPS)

print("Ma tran sau khi chay:")
print(final_grid)
print()
print(f"So buoc di : {steps}")
print(f"Bui da hut : {cleaned} / {total_dust} o")
print(f"Trang thai : {status}")
print(f"Ly do      : {reason}")